***
***
#  Przetwarzanie danych kategorialnych


Rzeczywiste zestawy danych zawierają często przynajmniej jedną kolumnę cech kategorialnych (nienumerycznych) - zwykle  etykiety klas są tego typu. Ponadto wiele algorytmów uczenia maszynowego wymaga, aby etykiety klas były kodowane w postaci liczb całkowitych. 

***
***

## Źródło kodów :

Wszystkie istotne kody umieszczone poniżej pochodzą z  plików Jupyter związanych z pozycją [1] bibliografii. Są one dostępne m.in. w repozytorium [https://github.com/rasbt/python-machine-learning-book-3rd-edition](https://github.com/rasbt/python-machine-learning-book-3rd-edition)

In [2]:
import numpy as np
from IPython.display import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import pandas as pd
import sys
from IPython.display import display, Math, Latex
import sympy as sym

## 1. Cechy nominalne i porządkowe

Dane kategorialne dzielimy na dwie grupy:  cechy \textit{nominalne}
i \textit{porządkowe}. 
\begin{enumerate}
\item Cechy porządkowe to takie wartości kategoryzujące, które możemy wykorzystać
do sortowania lub porządkowania informacji. Przykładowo rozmiar koszulki to cecha porządkowa,
ponieważ możemy określić kolejność rozmiarów XL > L > M. 
\item Cechy nominalne nie definiują kolejności próbek. Przykładowo, można
za tego typu cechę uznać kolor koszulki, ponieważ zazwyczaj stwierdzenie, że czerwony
jest większy od niebieskiego, nie ma sensu.
\end{enumerate}

Poniższa tabela \textit{DataFrame}  składa się z cechy nominalnej (Kolor), porządkowej
(Rozmiar) i numerycznej (Cena). W ostatniej kolumnie są przechowywane etykiety klas (załóżmy, że
stworzyliśmy zestaw danych przeznaczonych do uczenia nadzorowanego). 

In [3]:
df = pd.DataFrame([['Zielony', 'M', 10.1, 'klasa2'],
                   ['Czerwony', 'L', 13.5, 'klasa1'],
                   ['Niebieski', 'XL', 15.3, 'klasa2']])

df.columns = ['Kolor', 'Rozmiar', 'Cena', 'Etykieta klas']
df

,Kolor,Rozmiar,Cena,Etykieta klas
0,Zielony,M,10.1,klasa2
1,Czerwony,L,13.5,klasa1
2,Niebieski,XL,15.3,klasa2


<br>
<br>

## 2. Mapowanie cech porządkowych

Aby się upewnić, że algorytm uczenia poprawnie interpretuje cechy porządkowe, musimy przekształcić
wartości kategorialne występujące jako ciągi znaków w postać liczb całkowitych. Niestety,
nie istnieje żadna funkcja, która automatycznie przeniosłaby właściwy szyk etykiet z kolumny
Rozmiar, dlatego musimy własnoręcznie zdefiniować mapowanie cech. W poniższym przykładzie
zakładamy, że znamy różnice numeryczne pomiędzy cechami, np. XL = L+1 = M+2:

In [4]:
size_mapping = {'XL': 3,
                'L': 2,
                'M': 1}

df['Rozmiar'] = df['Rozmiar'].map(size_mapping)
df

,Kolor,Rozmiar,Cena,Etykieta klas
0,Zielony,1,10.1,klasa2
1,Czerwony,2,13.5,klasa1
2,Niebieski,3,15.3,klasa2


Przekształenie liczb całkowitych z powrotem na ich znakową reprezentację
(w którymś z późniejszych etapów) można wykonać  definiując słownik do odwrotnego mapowania

\textit{inv_size_mapping = \{v: k for k, v in size_mapping.items()\}}. 

Następnie  można go zastosować za pomocą metody $map$ do zmodyfikowanej kolumny cech:

In [5]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['Rozmiar'].map(inv_size_mapping)

0     M
1     L
2    XL
Name: Rozmiar, dtype: object

## 3. Kodowanie etykiet klas

Większość estymatorów klasyfikacji w interfejsie scikit-learn zawiera
wewnętrzne mechanizmy przekształcające etykiety klas w liczby całkowite, jednak zawsze
dobrym rozwiązaniem jest własnoręczne przygotowanie tych etykiet jako tablic z liczbami
całkowitymi w celu uniknięcia potencjalnych problemów technicznych. Do kodowania etykiet
klas można wykorzystać mechanizm podobny do
mapowania cech porządkowych. Należy pamiętać, że etykiety klas nie są cechami porządkowymi
i nie ma znaczenia, jaką liczbę całkowitą przydzielimy do danego ciągu znaków.
Możemy więc zwyczajnie ponumerować etykiety klas, począwszy od wartości 0:

In [6]:
# tworzy słownik mapowania
# przekształcający etykiety klas z ciągów znaków do postaci liczb całkowitych
class_mapping = {label: idx for idx, label in enumerate(np.unique(df['Etykieta klas']))}
class_mapping

{'klasa1': 0, 'klasa2': 1}

Teraz możemy wykorzystać słownik mapowania do przekształcenia etykiet klas w liczby całkowite:

In [7]:
# przekształca etykiety klas z ciągów znaków do postaci liczb całkowitych
df['Etykieta klas'] = df['Etykieta klas'].map(class_mapping)
df

,Kolor,Rozmiar,Cena,Etykieta klas
0,Zielony,1,10.1,1
1,Czerwony,2,13.5,0
2,Niebieski,3,15.3,1


Możemy odwrócić parę klucz – wartość w słowniku mapowania, aby przywrócić etykietom
klas pierwotne wartości w postaci ciągów znaków:

In [8]:
# odwraca proces mapowania
inv_class_mapping = {v: k for k, v in class_mapping.items()}
df['Etykieta klas'] = df['Etykieta klas'].map(inv_class_mapping)
df

,Kolor,Rozmiar,Cena,Etykieta klas
0,Zielony,1,10.1,klasa2
1,Czerwony,2,13.5,klasa1
2,Niebieski,3,15.3,klasa2


Uzyskamy ten sam efekt, gdy skorzystamy z wygodnej klasy LabelEncoder zaimplementowanej
bezpośrednio w bibliotece scikit-learn:

In [9]:
from sklearn.preprocessing import LabelEncoder

# kodowanie etykiet za pomocą klasy LabelEncoder
class_le = LabelEncoder()
y = class_le.fit_transform(df['Etykieta klas'].values)
y

array([1, 0, 1])

Metoda $fit\_transform$ stanowi jedynie skrót do oddzielnego wywołania
metod $fit$ i $transform$, a metodę $inverse\_transform$ możemy wykorzystać do przekształcania
liczb całkowitych z powrotem do reprezentacji etykiet klas w postaci ciągów znaków:

In [10]:
# odwrotne mapowanie
class_le.inverse_transform(y)

array(['klasa2', 'klasa1', 'klasa2'], dtype=object)

## 4. Kodowanie „gorącojedynkowe” cech nominalnych

Próbki znajdujące się w nominalnej kolumnie \textit{Kolor} trzeba przekształcić inaczej, gdyż zastowanie powyższych technik, tzn. 

In [11]:
X = df[['Kolor', 'Rozmiar', 'Cena']].values
color_le = LabelEncoder()
X[:, 0] = color_le.fit_transform(X[:, 0])#przekształcamy zerową kolumnę, czyli dane nominalne
X

array([[2, 1, 10.1],
       [0, 2, 13.5],
       [1, 3, 15.3]], dtype=object)

spowoduje, że po uruchomieniu powyższego kodu pierwsza kolumna w tabeli NumPy przechowuje teraz nowe
wartości cechy Kolor, które zostają zakodowane w następujący sposób:
Niebieski = 1, Zielony = 2, Czerwony = 0.

Gdybyśmy w tym momencie się zatrzymali i przesłali tabelę do klasyfikatora, popełnilibyśmy
jeden z najczęstszych błędów spotykanych podczas przetwarzania danych kategoryzujących.Mimo że wartości kolorów nie pojawiają się w określonej kolejności, algorytm uczący będzie teraz zakładał, że zielony jest większy od niebieskiego,
a niebieski od czerwonego. To założenie jest niewłaściwe, a jednak algorytm może generować
poprawne wyniki, ale mogą być one  dalekie od optymalnych.

Popularnym rozwiązaniem tego problemu jest zastosowanie techniki zwanej kodowaniem
„gorącojedynkowym” (ang. one-hot encoding). Koncepcją kryjącą się za tą metodą jest wprowadzenie
sztucznej cechy (ang. dummy feature) dla każdej unikatowej wartości w kolumnie cechy
nominalnej. W naszym przypadku możemy przekonwertować cechę Kolor w trzy nowe cechy:
Niebieski, Zielony i Czerwony. Wykorzystamy wartości binarne do wskazywania danego koloru
przykładu; np. niebieski przykład może zostać zakodowany jako Niebieski=1, Zielony=0,
Czerwony=0. Aby dokonać takiej transformacji, możemy wykorzystać klasę $OneHotEncoder$ zaimplementowaną
w module scikit-learn.preprocessing:

In [12]:
from sklearn.preprocessing import OneHotEncoder

X = df[['Kolor', 'Rozmiar', 'Cena']].values
color_ohe = OneHotEncoder()
color_ohe.fit_transform(X[:, 0].reshape(-1, 1)).toarray()

array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]])

In [13]:
from sklearn.compose import ColumnTransformer

X = df[['Kolor', 'Rozmiar', 'Cena']].values
c_transf = ColumnTransformer([ ('onehot', OneHotEncoder(), [0]),
                               ('nothing', 'passthrough', [1, 2])])
c_transf.fit_transform(X).astype(float)

array([[ 0. ,  0. ,  1. ,  1. , 10.1],
       [ 1. ,  0. ,  0. ,  2. , 13.5],
       [ 0. ,  1. ,  0. ,  3. , 15.3]])

In [14]:
# kodowanie gorącojedynkowe za pomocą biblioteki pandas

pd.get_dummies(df[['Kolor', 'Rozmiar', 'Cena']])

,Rozmiar,Cena,Kolor_Czerwony,Kolor_Niebieski,Kolor_Zielony
0,1,10.1,0,0,1
1,2,13.5,1,0,0
2,3,15.3,0,1,0


In [15]:
# ochrona współliniowości w funkcji get_dummies

pd.get_dummies(df[['Kolor', 'Rozmiar', 'Cena']], drop_first=True)

,Rozmiar,Cena,Kolor_Niebieski,Kolor_Zielony
0,1,10.1,0,1
1,2,13.5,0,0
2,3,15.3,1,0


In [16]:
# ochrona współiniowości dla klasy OneHotEncoder

color_ohe = OneHotEncoder(categories='auto', drop='first')
c_transf = ColumnTransformer([ ('onehot', color_ohe, [0]),
                               ('nothing', 'passthrough', [1, 2])])
c_transf.fit_transform(X).astype(float)

array([[ 0. ,  1. ,  1. , 10.1],
       [ 0. ,  0. ,  2. , 13.5],
       [ 1. ,  0. ,  3. , 15.3]])

<br>
<br>

## 5. Kodowanie cech porządkowych

Jeżeli nie jesteśmy pewni różnic numerycznych pomiędzy kategoriami cech porządkowych lub jeśli nie została zdefiniowana różnica pomiędzy dwiema cechami porządkowymi, możemy je kodować za pomocą kodowania progowego przy użyciu wartości 0/1. Na przykład możemy rozdzielić cechę "Rozmiar" mającą wartości M, L i XL na dwie nowe cechy: "x > M" i "x > L". Zacznijmy od naszej pierwotnej ramki danych:

In [17]:
df = pd.DataFrame([['Zielony', 'M', 10.1, 'klasa2'],
                   ['Czerwony', 'L', 13.5, 'klasa1'],
                   ['Niebieski', 'XL', 15.3, 'klasa2']])

df.columns = ['Kolor', 'Rozmiar', 'Cena', 'Etykieta klas']
df

,Kolor,Rozmiar,Cena,Etykieta klas
0,Zielony,M,10.1,klasa2
1,Czerwony,L,13.5,klasa1
2,Niebieski,XL,15.3,klasa2


Możemy użyć metody `apply` do tworzenia niestandardowych wyrażeń lambda po to, aby kodować te zmienne za pomocą wartości progowych:

In [18]:
df['x > M'] = df['Rozmiar'].apply(lambda x: 1 if x in {'L', 'XL'} else 0)
df['x > L'] = df['Rozmiar'].apply(lambda x: 1 if x == 'XL' else 0)

del df['Rozmiar']
df

,Kolor,Cena,Etykieta klas,x > M,x > L
0,Zielony,10.1,klasa2,0,0
1,Czerwony,13.5,klasa1,1,0
2,Niebieski,15.3,klasa2,1,1
